# Lab 05: GAN on MNIST

**Module 05** | Read `notes/05-gan-fundamentals.pdf` before running this notebook.

Train a minimal GAN on MNIST. Monitor generator and discriminator losses and inspect generated digit grids.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Generative Adversarial Networks (GANs) pit two networks against each other. The **generator** maps random noise to synthetic images; the **discriminator** tries to distinguish real images from fakes. The generator improves only when it fools the discriminator, an adversarial signal with no direct pixel-wise target.


We train on MNIST because the 28x28 grayscale domain is small enough for a fully connected GAN to produce recognizable digits within a few epochs.


### Step 1: Load MNIST

MNIST lives under `datasets/mnist` after running the course setup script. Images are normalized to `[-1, 1]` with `Tanh` on the generator output so real and fake tensors share the same range, a practical requirement for stable GAN training.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
BATCH_SIZE = 128
LATENT_DIM = 100
EPOCHS = 8
LR = 2e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"MNIST train images: {len(train_set):,}")
print(f"Batches per epoch:  {len(loader)}")
print(f"Image shape:        {train_set[0][0].shape}  (channels, height, width)")
print(f"Pixel range:        [{train_set[0][0].min():.1f}, {train_set[0][0].max():.1f}]  (normalized)")


### Step 2: Visualize real training digits

Before training any model, look at real data. The discriminator must learn what real digit structure looks like so it can reject blurry or noisy fakes.


In [ ]:
real_batch, real_labels = next(iter(loader))
sample_grid = make_grid(real_batch[:16].cpu(), nrow=4, normalize=True, value_range=(-1, 1))

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(sample_grid.permute(1, 2, 0).squeeze(), cmap="gray")
ax.set_title("16 real MNIST digits")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Labels for the 16 samples above:", real_labels[:16].tolist())


### Step 3: Generator and discriminator

The generator is a multilayer perceptron that expands noise through hidden layers to 784 pixels, reshaped to 1x28x28. The discriminator flattens the image and outputs a single probability of being real.

`LeakyReLU` activations are standard in shallow GANs because they avoid dead gradients on the negative side compared with plain ReLU.


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim: int = 100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")


### Step 4: Adversarial training loop

Each iteration updates the discriminator on real and fake batches, then updates the generator with labels flipped to "real", the standard trick so the generator receives a useful gradient through the discriminator.

We track generator loss, discriminator loss, and **discriminator accuracy** (how often D correctly classifies real as real and fake as fake). We also print mean D(real) and D(fake) from the last batch each epoch. Ideal training keeps these scores separated but not collapsed to 0 or 1.


In [ ]:
g_losses, d_losses = [], []
d_acc_real_hist, d_acc_fake_hist = [], []

for epoch in range(1, EPOCHS + 1):
    g_epoch, d_epoch = 0.0, 0.0
    d_correct_real, d_total_real = 0, 0
    d_correct_fake, d_total_fake = 0, 0
    last_d_real_mean, last_d_fake_mean = 0.0, 0.0

    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)
        real_labels = torch.ones(batch, 1, device=device)
        fake_labels = torch.zeros(batch, 1, device=device)

        # Discriminator
        D.zero_grad()
        d_real = D(real)
        loss_d_real = criterion(d_real, real_labels)
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z).detach()
        d_fake = D(fake)
        loss_d_fake = criterion(d_fake, fake_labels)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()

        d_correct_real += ((d_real > 0.5).float().sum().item())
        d_total_real += batch
        d_correct_fake += ((d_fake < 0.5).float().sum().item())
        d_total_fake += batch
        last_d_real_mean = d_real.mean().item()
        last_d_fake_mean = d_fake.mean().item()

        # Generator
        G.zero_grad()
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z)
        d_out = D(fake)
        loss_g = criterion(d_out, real_labels)
        loss_g.backward()
        opt_g.step()

        g_epoch += loss_g.item()
        d_epoch += loss_d.item()

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / len(loader))
    acc_real = d_correct_real / d_total_real
    acc_fake = d_correct_fake / d_total_fake
    d_acc_real_hist.append(acc_real)
    d_acc_fake_hist.append(acc_fake)

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"G loss {g_losses[-1]:.4f}  D loss {d_losses[-1]:.4f}  "
        f"D acc real {acc_real * 100:.1f}%  fake {acc_fake * 100:.1f}%"
    )
    print(
        f"  Last-batch mean D(real)={last_d_real_mean:.3f}  "
        f"D(fake)={last_d_fake_mean:.3f}  "
        f"(want real high, fake low)"
    )

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), g_losses, label="Generator", marker="o")
plt.plot(range(1, EPOCHS + 1), d_losses, label="Discriminator", marker="s")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("GAN training losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 5: Qualitative comparison of real vs. generated digits

Side-by-side grids make evaluation immediate. Real digits come from the training set; fakes are fresh noise passed through the generator after training.


In [ ]:
real_batch, _ = next(iter(loader))
real_batch = real_batch[:64].to(device)

G.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=device)
    fake_batch = G(z)

real_grid = make_grid(real_batch.cpu(), nrow=8, normalize=True, value_range=(-1, 1))
fake_grid = make_grid(fake_batch.cpu(), nrow=8, normalize=True, value_range=(-1, 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(real_grid.permute(1, 2, 0).squeeze(), cmap="gray")
axes[0].set_title("Real MNIST")
axes[0].axis("off")
axes[1].imshow(fake_grid.permute(1, 2, 0).squeeze(), cmap="gray")
axes[1].set_title("Generated digits")
axes[1].axis("off")
plt.tight_layout()
plt.show()


### Step 6: Interpret the results

Read the numbers and images together. A healthy GAN shows D(real) above D(fake), both losses oscillating rather than collapsing, and generated digits that resemble strokes even if they are blurry.


In [ ]:
with torch.no_grad():
    probe_real = next(iter(loader))[0][:128].to(device)
    probe_fake = G(torch.randn(128, LATENT_DIM, device=device))
    mean_real = D(probe_real).mean().item()
    mean_fake = D(probe_fake).mean().item()

print("=" * 60)
print("Lab 05 interpretation")
print("=" * 60)
print(f"  Final G loss:          {g_losses[-1]:.4f}")
print(f"  Final D loss:          {d_losses[-1]:.4f}")
print(f"  D accuracy on real:    {d_acc_real_hist[-1] * 100:.1f}%")
print(f"  D accuracy on fake:    {d_acc_fake_hist[-1] * 100:.1f}%")
print(f"  Mean D(real) probe:    {mean_real:.3f}  (closer to 1.0 is confident real)")
print(f"  Mean D(fake) probe:    {mean_fake:.3f}  (closer to 0.0 is confident fake)")
print()
if mean_real > mean_fake + 0.1:
    print("  Discriminator separates real and fake. Generator is learning.")
elif mean_real < 0.6 and mean_fake < 0.6:
    print("  Both scores are low. Discriminator may be winning too strongly.")
else:
    print("  Scores are close. Training may still be balancing.")
print("  Inspect the grid: digit-like blobs mean G is capturing stroke structure.")
print("=" * 60)
